### Accumulators
<ul>
    <li>Accumulators are updateable variables that are added through an associative &
commutative operation on all nodes. </li>
    <li>They are used to implement counters or sums</li>
    <li>Spark natively supports numeric accumulators. </li>
</ul>

### Spark AQE (Adaptive Query Execution)
<p><b>Adaptive Query Execution (AQE)</b> is an optimization technique in Spark SQL that makes
use of the runtime statistics to choose the most efficient query execution plan. </p>
<p>In Spark 3.0, the AQE framework is shipped with three features:</p>
<ul>
    <li>Dynamically coalescing shuffle partitions (were the are empty partitions AQE tries to coalesc it)</li>
    <li>Dynamically switching join strategies (convert sort merge join to broadcast join)</li>
    <li>Dynamically optimizing skew joins (were few partition has enoromous data othors don't AQE tries to break it and make the huge partition into smaller partition) </li>
</ul>

#### In PySpark
<pre>
spark.conf.set("spark.sql.adaptive.enabled", "true")
</pre>

#### When submitting a Spark job
<pre>
--conf spark.sql.adaptive.enabled=true
</pre>

### Key Configuration Parameters for Adaptive Query Execution

| Configuration Key                                 | Description                                                                 |
|---------------------------------------------------|-----------------------------------------------------------------------------|
| `spark.sql.adaptive.enabled`                     | Enables Adaptive Query Execution (AQE). Must be set to `true`.             |
| `spark.sql.adaptive.coalescePartitions.enabled`  | Dynamically reduces the number of shuffle partitions based on size.        |
| `spark.sql.adaptive.skewJoin.enabled`            | Detects and handles skewed joins by splitting large partitions.            |
| `spark.sql.adaptive.localShuffleReader.enabled`  | Uses local shuffle reader to reduce data movement across nodes.            |
| `spark.sql.adaptive.joinSelection.enabled`       | Dynamically selects join strategies (e.g., broadcast vs. shuffle).         |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes`| Target size for coalesced shuffle partitions (default: 64MB).              |
| `spark.sql.adaptive.maxNumPostShufflePartitions` | Upper bound on the number of post-shuffle partitions.                      |

## Memory Calculation

Melwin go throught the docs

### Types of Partitioning in Spark
<ul>
    <li>Hash Partitioning</li>
    <li>Range Partitioning</li>
</ul>

#### Hash Partitioning
<ul>
    <li>Hash Partitioning attempts to spread the data evenly across various partitions based
on the key.</li>
    <li> Object.hashCode method is used to determine the partition in Spark as
partition = key.hashCode ( ) % numPartitions.</li>
</ul>

#### Range Partitioning
<ul>
    <li>Some Spark RDDs have keys that follow a particular ordering for such RDDs range
partitioning is an efficient partitioning technique.</li>
    <li>In range partitioning method, tuples having keys within the same range will appear on
the same machine.</li>
    <li>Keys in a range partitioner are partitioned based on the set of sorted range of keys and
ordering of keys.</li>
</ul>

## Cache and Persist

its Generally Used on Repeated access to the same preprocessed data. Avoid recomputing expensive joins or filters. Speed up exploratory queries. and we need not recaluclate the data

## Cache

Cache stores in memory ( in deseralized format ) + disk (in searlized format) <br>
 if memory is not enough then the rest of the data which needs to be cached will be stored in disk <br>(it will be CPU intensive as in memoery it stores in desearalized and in disk its searalized )

example 

<pre>
df_cached = df.filter(df["status"] == "active").cache()
df_cached.count()  # triggers caching

</pre>


### Persis 

there are many memory options 
only Memory

<table>
    <tr>
        <th>MEMORY_ONLY</th>
        <td>Stores only in memory. If not enough space, recomputes.</td>
    </tr>
    <tr>
        <th>MEMORY_AND_DISK</th>
        <td>Stores in memory, spills to disk if needed. Default for DataFrames.</td>
    </tr>
    <tr>
        <th>DISK_ONLY</th>
        <td>Stores only on disk. Useful for large datasets.</td>
    </tr>
    <tr>
        <th>MEMORY_ONLY_SER</th>
        <td>Stores serialized objects in memory (less space, slower access).</td>
    </tr>
    <tr>
        <th>MEMORY_AND_DISK_SER</th>
        <td>Serialized in memory, spills to disk.</td>
    </tr>
</table>    

example 
<pre>
from pyspark import StorageLevel

df_persisted = df.filter(df["status"] == "active").persist(StorageLevel.MEMORY_AND_DISK)
df_persisted.count()

</pre>

### Cache vs Persist

| Feature              | `cache()`                              | `persist()`                                 |
|----------------------|----------------------------------------|---------------------------------------------|
| **Default Behavior** | Uses `MEMORY_AND_DISK` for DataFrames  | User-defined storage level                  |
| **Customizable?**    | ❌ No                                   | ✅ Yes — supports multiple storage levels    |
| **Syntax**           | `df.cache()`                           | `df.persist(StorageLevel.MEMORY_AND_DISK)`  |
| **Use Case**         | Quick reuse with default settings      | Fine-grained control over memory/disk usage |
| **Storage Options**  | Fixed (memory + disk fallback)         | Flexible: memory-only, disk-only, serialized |
| **Fault Tolerance**  | Recomputes if partition is lost        | Depends on chosen storage level             |
| **Monitoring**       | Visible in Spark UI (Storage tab)      | Same — tracked by Spark                     |




### Catalyst Optimization
<ul>
    <li>Spark SQL is an Apache Spark module for structured data processing</li>
    <li>One of the big differences with the Spark API RDD is that its interfaces provide
additional information to perform more efficient processes.</li>
    <li>This information is also useful for Spark SQL to benefit internally from using its
Catalyst optimizer and improve performance in data processing.</li>
</ul>

#### What is Catalyst?
<ul>
    <li>Catalyst is Spark’s <b>query optimization framework</b>, built using <b>functional programming in Scala</b>.<br> It powers both SQL queries and DataFrame/Dataset operations by converting them into optimized execution plans.
</li>
    <li>Its two main purposes are: <br>
        <b>first</b>, to add new optimization techniques to solve some problems with “big data” <br>
        <b>second</b>, to allow developers to expand and customize the functions of the optimizer.
    </li>
</ul>

#### Using Catalyst in Spark SQL
<p>The Catalyst Optimizer in Spark offers <b>rule-based</b> and <b>cost-based</b> optimization.<br> 
    <b>Rule-based optimization</b> indicates how to execute the query from a set of defined rules. Meanwhile,<br>
    <b>Cost-based optimization</b> generates multiple execution plans and compares them to choose the lowest cost one. 
</p>

#### The four phases of the transformation that Catalyst performs
<ul>
    <li>Analysis</li>
    <li>Logic Optimization Plan</li>
    <li>Physical plan</li>
    <li>Code generation</li>
</ul>

### Catalyst Optimization Pipeline

<b>Here’s how Catalyst processes a query step by step:</b>
<ol>
<li>Parsing</li>
    <ul>
<li>    - Converts your SQL or DataFrame code into an Abstract Syntax Tree (AST).</li>
<li>    - Think of this as the raw blueprint of your query.</li>
        </ul>
<li>Analysis</li>
    <ul>
<li>    - Resolves references: column names, table names, functions.</li>
<li>    - Validates the query structure and semantics.</li>
<li>    - Example: Ensures df.select("amount") actually refers to a valid column.</li>
        </ul>
<li>Logical Optimization</li>
    <ul>
<li>    - Applies rule-based transformations to improve the logical plan.</li>
<li>    - Examples:</li>
<li>    - <b>Predicate pushdown</b>: Move filters closer to the data source.</li>
        <li>    - <b>Constant folding</b>: Precompute static expressions.</li>
<li>    - <b>Projection pruning</b>: Remove unused columns.</li>
        </ul>
<li>Physical Planning</li>
    <ul>
<li>    - Converts the optimized logical plan into one or more physical plans.</li>
<li>    - Each physical plan represents a different strategy for execution (e.g., broadcast join vs. shuffle join).</li>
        </ul>
<li>Cost-Based Optimization (CBO)</li>
    <ul>
<li>    - If enabled, Spark uses statistics (like row counts, column cardinality) to choose the most efficient physical plan.</li>
<li>    - Example: Choosing a broadcast join if one table is small.</li>
        </ul>
<li>Code Generation</li>
    <ul>
<li>    - Spark uses whole-stage code generation to compile parts of the query into Java bytecode.</li>
<li>    - This minimizes JVM overhead and speeds up execution.</li>
        </ul>
</ol>

### Predicate and Projection Pushdown in Spark

### Predicate vs Projection Pushdown in Spark 3 


https://towardsdatascience.com/predicate-vs-projection-pushdown-in-spark-3-ac24c4d11855#:~:text=Projection%20Pushdown%20stands%20for%20the,those%20columns%20will%20be%20returned.

### What is a predicate pushdown?
<p>The basic idea of predicate pushdown is that certain parts of SQL queries (the predicates) can be “pushed” to where the data lives. <br>This optimization can drastically reduce query/processing time by filtering out data earlier rather than later.</p>